# 06 · Experiment Results — Paper Tables and Figures

Part of the study *Multi-Label 12-Lead ECG Classification on the PTB-XL Dataset* ([arXiv:2609.12803](https://arxiv.org/abs/2609.12803)). Run order, folder layout, and results: [project README](../README.md).

Reads `results/<model>/seed_<seed>/` (canonical protocol only). Ensemble methods are included even if notebook `05` has not been run: equal / weighted / stacking are computed from member predictions (weights and stacker fit on fold 9 only, then applied to **test fold 10**).

**What this notebook shows**

1. **Dataset figures** — superclass prevalence and age–sex pyramid (diagnostic-labelled PTB-XL 1.0.3)
2. Journal-style **leaderboard** with every protocol macro metric (AUROC, AUPRC, F1, micro-F1, Precision, Recall, Specificity, Hamming loss, subset accuracy)
3. **Per-class** AUROC / AUPRC / F1 / Precision / Sensitivity / Specificity, plus 2×2 counts (TN/FP/FN/TP)
4. **Literature overlay** vs Strodthoff et al. 2021 PTB-XL *superdiagnostic* (5 superclass) test numbers
5. ROC / PR, diversity, and per-class 2×2 confusion matrices

Primary metric is **macro-AUPRC**. Three seeds → mean ± SD. No 5×5 multiclass confusion matrix. Reported split is **test fold 10**; thresholds remain val F1-max.


In [ ]:
!pip install -q openpyxl scikit-learn pandas numpy matplotlib seaborn


In [ ]:
from google.colab import drive
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import Markdown, display
from sklearn.metrics import precision_recall_curve, roc_curve, auc

drive.mount('/content/drive')
PROJECT_DIR = Path('/content/drive/MyDrive') / 'ECG AI'
sys.path.insert(0, str(PROJECT_DIR / 'notebooks'))

from ecg_common import (
    CLASSES,
    ENSEMBLE_MEMBERS,
    EVALUATE_TEST as DEFAULT_EVALUATE_TEST,
    SEEDS,
    find_ptbxl_data_root,
    align_prediction_frames,
    apply_thresholds,
    class_auprc_weights,
    compute_multilabel_metrics,
    disagreement_rate,
    display_model_name,
    dump_json,
    ensure_dir,
    equal_soft_vote,
    error_overlap,
    fit_stacker,
    load_json,
    load_predictions,
    optimize_thresholds_f1,
    pairwise_prediction_correlation,
    per_class_confusion_2x2,
    predict_stacker,
    seed_results_dir,
    threshold_vector,
    weighted_soft_vote,
)

EVALUATE_TEST = True
SPLIT = 'test'
RESULTS_DIR = PROJECT_DIR / 'results'
DATA_DIR = PROJECT_DIR / 'data'
OUT_DIR = ensure_dir(PROJECT_DIR / 'experiments' / 'comparison' / 'latest')
FIG_DIR = ensure_dir(PROJECT_DIR / 'paper' / 'results_figures')
TABLE_DIR = ensure_dir(PROJECT_DIR / 'paper' / 'results_tables')
DATASET_FIG_DIR = ensure_dir(PROJECT_DIR / 'paper' / 'dataset_figures')

MEMBER_KEYS = list(ENSEMBLE_MEMBERS)
ENSEMBLE_KEYS = ['equal_soft_voting', 'weighted_soft_voting', 'stacking']
ALL_KEYS = MEMBER_KEYS + ENSEMBLE_KEYS

MM_IN = 1.0 / 25.4
COL_SINGLE = 89 * MM_IN
COL_DOUBLE = 183 * MM_IN

OKABE = ['#0072B2', '#D55E00', '#E69F00', '#009E73', '#CC79A7', '#56B4E9', '#000000', '#882255']
MODEL_COLOR = {k: OKABE[i] for i, k in enumerate(ALL_KEYS)}
CLASS_COLOR = {name: OKABE[i] for i, name in enumerate(CLASSES)}

plt.rcParams.update({
    'font.family': 'DejaVu Sans',
    'font.size': 9,
    'axes.labelsize': 9,
    'axes.titlesize': 10,
    'xtick.labelsize': 8,
    'ytick.labelsize': 8,
    'legend.fontsize': 8,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.linewidth': 0.7,
    'pdf.fonttype': 42,
    'ps.fonttype': 42,
    'savefig.bbox': 'tight',
    'savefig.pad_inches': 0.04,
    'figure.dpi': 120,
    'axes.facecolor': 'white',
    'figure.facecolor': 'white',
})

print('RESULTS_DIR', RESULTS_DIR)
print('split', SPLIT, '| EVALUATE_TEST', EVALUATE_TEST, '| seeds', SEEDS)


## Dataset — prevalence and age–sex

Diagnostic-labelled PTB-XL 1.0.3 records (empty diagnostic dropped). Same figures as notebook `00` paper section: superclass prevalence and the age–sex pyramid.


In [ ]:
import ast
from sklearn.preprocessing import MultiLabelBinarizer

CLASS_COLOR = {name: OKABE[i] for i, name in enumerate(CLASSES)}


def _parse_scp_codes(value):
    if isinstance(value, dict):
        return value
    return ast.literal_eval(value)


def load_diagnostic_metadata(data_root: Path) -> pd.DataFrame:
    meta = pd.read_csv(data_root / 'ptbxl_database.csv', index_col='ecg_id')
    meta['scp_codes'] = meta['scp_codes'].apply(_parse_scp_codes)
    agg = pd.read_csv(data_root / 'scp_statements.csv', index_col=0)
    agg = agg[agg['diagnostic'] == 1.0]
    def supers(codes):
        found = []
        for code in codes.keys():
            if code not in agg.index:
                continue
            cls = agg.loc[code, 'diagnostic_class']
            if pd.notna(cls) and str(cls) in CLASSES:
                found.append(str(cls))
        return [name for name in CLASSES if name in set(found)]
    meta['diagnostic_superclass'] = meta['scp_codes'].apply(supers)
    meta = meta[meta['diagnostic_superclass'].map(len) > 0].copy()
    return meta


def encode_y(labels) -> np.ndarray:
    mlb = MultiLabelBinarizer(classes=list(CLASSES))
    return mlb.fit_transform(labels).astype(np.int32)


def save_dataset_fig(fig, stem: str):
    for folder in (DATASET_FIG_DIR, FIG_DIR):
        fig.savefig(folder / f'{stem}.pdf')
        fig.savefig(folder / f'{stem}.png', dpi=300)
    display(fig)
    plt.close(fig)
    print('saved', stem, '->', DATASET_FIG_DIR)


def plot_superclass_prevalence(y: np.ndarray):
    n = int(y.shape[0])
    counts = y.sum(axis=0).astype(int)
    fig, ax = plt.subplots(figsize=(COL_SINGLE, 2.4))
    ypos = np.arange(len(CLASSES))
    ax.barh(ypos, counts, color=[CLASS_COLOR[n] for n in CLASSES], height=0.72, edgecolor='none')
    ax.set_yticks(ypos, CLASSES)
    ax.invert_yaxis()
    ax.set_xlabel('Records (multi-label)')
    xmax = max(int(counts.max()) * 1.28, 1)
    ax.set_xlim(0, xmax)
    for i, count in enumerate(counts):
        pct = 100.0 * count / n
        ax.text(count + xmax * 0.01, i, f'{int(count):,}  ({pct:.1f}%)', va='center', fontsize=8)
    ax.set_title('Superclass prevalence', loc='left', fontweight='bold')
    fig.tight_layout()
    return fig, counts, n


def plot_age_sex_pyramid(frame: pd.DataFrame):
    bins = np.arange(0, 101, 5)
    centers = bins[:-1] + 2.5
    male = frame.loc[frame['sex'] == 0, 'age'].dropna().to_numpy()
    female = frame.loc[frame['sex'] == 1, 'age'].dropna().to_numpy()
    h_m, _ = np.histogram(male, bins=bins)
    h_f, _ = np.histogram(female, bins=bins)
    fig, ax = plt.subplots(figsize=(COL_SINGLE, 3.2))
    ax.barh(centers, -h_m, height=4.2, color='#0072B2', label='Male', edgecolor='none')
    ax.barh(centers, h_f, height=4.2, color='#CC79A7', label='Female', edgecolor='none')
    ax.set_xlabel('Records')
    ax.set_ylabel('Age (years)')
    ax.set_ylim(0, 100)
    xmax = int(max(h_m.max() if len(h_m) else 0, h_f.max() if len(h_f) else 0) * 1.15) or 1
    ax.set_xlim(-xmax, xmax)
    ax.xaxis.set_major_formatter(lambda x, _pos: f'{int(abs(x)):,}')
    ax.axvline(0, color='#334155', lw=0.6)
    ax.legend(frameon=False, loc='lower right')
    ax.text(0.02, 0.98, f'M {len(male):,}', transform=ax.transAxes, va='top', color='#0072B2', fontsize=8, fontweight='bold')
    ax.text(0.98, 0.98, f'F {len(female):,}', transform=ax.transAxes, va='top', ha='right', color='#CC79A7', fontsize=8, fontweight='bold')
    ax.set_title('Age–sex distribution', loc='left', fontweight='bold')
    fig.tight_layout()
    return fig


data_root = find_ptbxl_data_root(DATA_DIR)
meta = load_diagnostic_metadata(data_root)
y = encode_y(meta['diagnostic_superclass'].tolist())
n_records = int(y.shape[0])
n_patients = int(meta['patient_id'].nunique())
display(Markdown(
    f'**PTB-XL 1.0.3** diagnostic-labelled: **{n_records:,}** records / **{n_patients:,}** patients. '
    f'Split remains official `strat_fold` (train 1–8, val 9, test 10).'
))

fig_prev, counts, n = plot_superclass_prevalence(y)
save_dataset_fig(fig_prev, 'fig01_superclass_prevalence')

prev = pd.DataFrame({
    'class': list(CLASSES),
    'n': counts,
    'pct': np.round(100.0 * counts / n, 2),
})
display(prev.style.hide(axis='index').set_caption('Table D1. Superclass prevalence (multi-label)').set_table_styles([
    {'selector': 'caption', 'props': 'caption-side: top; font-weight: 700; text-align: left;'},
    {'selector': 'th', 'props': 'background-color: #1B2838; color: white; padding: 6px 8px;'},
    {'selector': 'td', 'props': 'padding: 5px 8px; text-align: center;'},
]))
prev.to_csv(TABLE_DIR / 'table_d1_prevalence.csv', index=False)

fig_age = plot_age_sex_pyramid(meta)
save_dataset_fig(fig_age, 'fig04_age_sex')

demo = pd.DataFrame({
    'group': ['Male (sex=0)', 'Female (sex=1)', 'Records', 'Patients'],
    'n': [
        int((meta['sex'] == 0).sum()),
        int((meta['sex'] == 1).sum()),
        n_records,
        n_patients,
    ],
})
display(demo.style.hide(axis='index').set_caption('Table D2. Age–sex cohort size').set_table_styles([
    {'selector': 'caption', 'props': 'caption-side: top; font-weight: 700; text-align: left;'},
    {'selector': 'th', 'props': 'background-color: #1B2838; color: white; padding: 6px 8px;'},
    {'selector': 'td', 'props': 'padding: 5px 8px; text-align: center;'},
]))
demo.to_csv(TABLE_DIR / 'table_d2_age_sex_counts.csv', index=False)
print('DATA_ROOT', data_root)


## Helpers

Collection, on-the-fly ensemble, journal tables, and shared plot functions.


In [ ]:
def _member_pred_path(key, seed, split):
    return seed_results_dir(RESULTS_DIR, key, seed) / f'{split}_predictions.csv'


def _ensemble_pred_path(key, seed, split):
    return (
        PROJECT_DIR / 'experiments' / 'ensemble' / f'seed_{seed}'
        / 'predictions' / f'{split}_{key}_predictions.csv'
    )


def _thr_path(key, seed):
    return seed_results_dir(RESULTS_DIR, key, seed) / 'thresholds.json'


def load_member_frames(seed, split):
    frames = {}
    for key in MEMBER_KEYS:
        path = _member_pred_path(key, seed, split)
        if not path.exists():
            return None
        frames[key] = pd.read_csv(path)
    return frames


def thresholds_for_member(key, seed, y_true, y_prob):
    path = _thr_path(key, seed)
    if path.exists():
        return threshold_vector(load_json(path), CLASSES)
    return threshold_vector(optimize_thresholds_f1(y_true, y_prob, CLASSES), CLASSES)


def ensemble_bundle(seed):
    """Val-fitted ensemble (weights, stacker, thresholds). Never fit on test."""
    val_frames = load_member_frames(seed, 'val')
    if val_frames is None:
        return None
    _, val_probs, y_val = align_prediction_frames(val_frames, CLASSES)
    weights = class_auprc_weights(val_probs, y_val, MEMBER_KEYS, CLASSES)
    stackers = fit_stacker(val_probs, y_val, MEMBER_KEYS, CLASSES)
    probs = {
        'equal_soft_voting': equal_soft_vote(val_probs, MEMBER_KEYS),
        'weighted_soft_voting': weighted_soft_vote(val_probs, weights, MEMBER_KEYS),
        'stacking': predict_stacker(stackers, val_probs, MEMBER_KEYS, CLASSES),
    }
    thr = {name: optimize_thresholds_f1(y_val, p, CLASSES) for name, p in probs.items()}
    return {
        'weights': weights,
        'stackers': stackers,
        'val_probs': val_probs,
        'y_val': y_val,
        'val_ens_probs': probs,
        'thresholds': thr,
        'val_frames': val_frames,
    }


def ensemble_split_probs(bundle, seed, split):
    if split == 'val':
        return bundle['y_val'], bundle['val_ens_probs']
    test_frames = load_member_frames(seed, 'test')
    if test_frames is None:
        return None, None
    _, test_probs, y_test = align_prediction_frames(test_frames, CLASSES)
    ens = {
        'equal_soft_voting': equal_soft_vote(test_probs, MEMBER_KEYS),
        'weighted_soft_voting': weighted_soft_vote(test_probs, bundle['weights'], MEMBER_KEYS),
        'stacking': predict_stacker(bundle['stackers'], test_probs, MEMBER_KEYS, CLASSES),
    }
    return y_test, ens


def load_model_arrays(key, seed, split, bundle=None):
    if key in MEMBER_KEYS:
        path = _member_pred_path(key, seed, split)
        if not path.exists():
            return None
        _, y_true, y_prob, _ = load_predictions(path, CLASSES)
        thr = thresholds_for_member(key, seed, y_true, y_prob)
        return y_true, y_prob, thr
    path = _ensemble_pred_path(key, seed, split)
    if path.exists():
        _, y_true, y_prob, _ = load_predictions(path, CLASSES)
        if bundle is None:
            bundle = ensemble_bundle(seed)
        thr = 0.5
        if bundle is not None:
            thr = threshold_vector(bundle['thresholds'][key], CLASSES)
        return y_true, y_prob, thr
    if bundle is None:
        bundle = ensemble_bundle(seed)
    if bundle is None:
        return None
    y_true, ens = ensemble_split_probs(bundle, seed, split)
    if y_true is None:
        return None
    return y_true, ens[key], threshold_vector(bundle['thresholds'][key], CLASSES)


MACRO_COLS = [
    'macro_auroc', 'macro_auprc', 'macro_f1', 'micro_f1',
    'precision', 'recall', 'specificity', 'hamming_loss', 'subset_accuracy',
]
CLASS_METRIC_KEYS = ('auroc', 'auprc', 'f1', 'precision', 'recall', 'sensitivity', 'specificity')
CM_COUNT_KEYS = ('tn', 'fp', 'fn', 'tp')
CLASS_VALUE_COLS = [f'{m}_{c}' for m in CLASS_METRIC_KEYS for c in CLASSES]
CM_VALUE_COLS = [f'cm_{c}_{k}' for c in CLASSES for k in CM_COUNT_KEYS]


def collect_leaderboard(split):
    rows = []
    per_seed = []
    for key in ALL_KEYS:
        seed_metrics = []
        for seed in SEEDS:
            loaded = load_model_arrays(key, seed, split)
            if loaded is None:
                continue
            y_true, y_prob, thr = loaded
            mets = compute_multilabel_metrics(y_true, y_prob, CLASSES, thresholds=thr)
            rec = {'key': key, 'seed': int(seed), **{c: float(mets[c]) for c in MACRO_COLS}}
            for cls in CLASSES:
                for m in CLASS_METRIC_KEYS:
                    rec[f'{m}_{cls}'] = float(mets[f'{m}_{cls}'])
                for k in CM_COUNT_KEYS:
                    rec[f'cm_{cls}_{k}'] = float(mets[f'cm_{cls}_{k}'])
            seed_metrics.append(rec)
            per_seed.append(rec)
        if not seed_metrics:
            continue
        df = pd.DataFrame(seed_metrics)
        row = {
            'key': key,
            'Model': display_model_name(key),
            'family': 'Ensemble' if key in ENSEMBLE_KEYS else 'Member',
            'n_seeds': int(len(df)),
        }
        for col in MACRO_COLS + CLASS_VALUE_COLS + CM_VALUE_COLS:
            row[col] = float(df[col].mean())
            row[f'{col}_sd'] = float(df[col].std(ddof=0)) if len(df) > 1 else 0.0
        rows.append(row)
    board = pd.DataFrame(rows)
    if not board.empty:
        order = {k: i for i, k in enumerate(ALL_KEYS)}
        board['_ord'] = board['key'].map(order)
        board = board.sort_values('_ord').drop(columns='_ord').reset_index(drop=True)
    return board, pd.DataFrame(per_seed)


def fmt_mean_sd(mean, sd, n, digits=3):
    if pd.isna(mean):
        return '—'
    if n > 1 and sd and sd > 0:
        return f'{mean:.{digits}f} ± {sd:.{digits}f}'
    return f'{mean:.{digits}f}'


def formatted_leaderboard(board):
    out = pd.DataFrame({
        'Model': board['Model'],
        'Family': board['family'],
        'Macro AUROC': [fmt_mean_sd(r.macro_auroc, r.macro_auroc_sd, r.n_seeds) for r in board.itertuples()],
        'Macro AUPRC': [fmt_mean_sd(r.macro_auprc, r.macro_auprc_sd, r.n_seeds) for r in board.itertuples()],
        'Macro F1': [fmt_mean_sd(r.macro_f1, r.macro_f1_sd, r.n_seeds) for r in board.itertuples()],
        'Micro F1': [fmt_mean_sd(r.micro_f1, r.micro_f1_sd, r.n_seeds) for r in board.itertuples()],
        'Precision': [fmt_mean_sd(r.precision, r.precision_sd, r.n_seeds) for r in board.itertuples()],
        'Recall': [fmt_mean_sd(r.recall, r.recall_sd, r.n_seeds) for r in board.itertuples()],
        'Specificity': [fmt_mean_sd(r.specificity, r.specificity_sd, r.n_seeds) for r in board.itertuples()],
        'Hamming ↓': [fmt_mean_sd(r.hamming_loss, r.hamming_loss_sd, r.n_seeds) for r in board.itertuples()],
        'Subset acc.': [fmt_mean_sd(r.subset_accuracy, r.subset_accuracy_sd, r.n_seeds) for r in board.itertuples()],
        'Seeds': board['n_seeds'],
    })
    return out


def _best_second_masks(values, higher=True):
    arr = np.asarray(values, dtype=float)
    valid = np.isfinite(arr)
    best = np.zeros(len(arr), dtype=bool)
    second = np.zeros(len(arr), dtype=bool)
    if valid.sum() == 0:
        return best, second
    order = np.argsort(arr[valid])
    if higher:
        order = order[::-1]
    idx = np.flatnonzero(valid)
    best[idx[order[0]]] = True
    if len(order) > 1:
        second[idx[order[1]]] = True
    return best, second


def save_fig(fig, stem):
    pdf = FIG_DIR / f'{stem}.pdf'
    png = FIG_DIR / f'{stem}.png'
    fig.savefig(pdf)
    fig.savefig(png, dpi=300)
    fig.savefig(OUT_DIR / f'{stem}.png', dpi=200)
    return png


def draw_journal_table(df, title, stem, numeric_source=None, higher_cols=None, lower_cols=None):
    """Publication table: navy header, ensemble shading, bold best / italic second."""
    higher_cols = set(higher_cols or [])
    lower_cols = set(lower_cols or [])
    n_row, n_col = df.shape
    fig_w = max(COL_DOUBLE, 0.95 * (n_col + 1) + 1.4)
    fig_h = 0.42 * (n_row + 2) + 0.55
    fig, ax = plt.subplots(figsize=(fig_w, fig_h))
    ax.axis('off')
    ax.set_title(title, loc='left', fontsize=11, fontweight='bold', pad=8, color='#1B2838')

    cell_text = df.astype(str).values.tolist()
    col_labels = list(df.columns)
    table = ax.table(
        cellText=cell_text,
        colLabels=col_labels,
        cellLoc='center',
        loc='upper center',
    )
    table.auto_set_font_size(False)
    table.set_fontsize(8)
    table.scale(1.0, 1.55)

    header_c = '#1B2838'
    member_c = '#F7F9FC'
    ensemble_c = '#E8F1FA'
    best_c = '#D9EAD3'
    second_c = '#FFF2CC'

    for j, label in enumerate(col_labels):
        cell = table[0, j]
        cell.set_facecolor(header_c)
        cell.set_text_props(color='white', fontweight='bold', fontsize=8)
        cell.set_edgecolor('#1B2838')
        if j == 0:
            cell._loc = 'left'

    family_col = df.columns.get_loc('Family') if 'Family' in df.columns else None
    for i in range(n_row):
        is_ens = False
        if family_col is not None:
            is_ens = str(df.iloc[i, family_col]).lower().startswith('ens')
        base = ensemble_c if is_ens else member_c
        for j in range(n_col):
            cell = table[i + 1, j]
            cell.set_facecolor(base)
            cell.set_edgecolor('#D0D5DD')
            if j == 0:
                cell._loc = 'left'
                cell.set_text_props(ha='left', fontweight='semibold')

    if numeric_source is not None:
        for col in list(higher_cols) + list(lower_cols):
            if col not in df.columns or col not in numeric_source.columns:
                continue
            j = df.columns.get_loc(col)
            higher = col in higher_cols
            best, second = _best_second_masks(numeric_source[col].to_numpy(), higher=higher)
            for i in range(n_row):
                cell = table[i + 1, j]
                if best[i]:
                    cell.set_facecolor(best_c)
                    cell.set_text_props(fontweight='bold')
                elif second[i]:
                    cell.set_facecolor(second_c)
                    cell.set_text_props(fontstyle='italic')

    fig.tight_layout()
    save_fig(fig, stem)
    display(fig)
    plt.close(fig)
    print('saved', stem)


def style_numeric_table(df, caption, higher=None, lower=None):
    higher = higher or []
    lower = lower or []
    styler = (
        df.style
        .format(precision=3)
        .set_caption(caption)
        .set_table_styles([
            {'selector': 'caption', 'props': 'caption-side: top; font-weight: 700; font-size: 13px; color: #1B2838; text-align: left; padding: 4px 0;'},
            {'selector': 'th', 'props': 'background-color: #1B2838; color: white; font-weight: 600; text-align: center; padding: 6px 8px;'},
            {'selector': 'td', 'props': 'padding: 5px 8px; text-align: center; border-bottom: 1px solid #E5E7EB;'},
            {'selector': 'th.col0, td.col0', 'props': 'text-align: left; font-weight: 600;'},
        ], overwrite=False)
        .set_properties(**{'font-size': '12px'})
    )
    if higher:
        styler = styler.background_gradient(subset=higher, cmap='Blues', axis=0)
        styler = styler.highlight_max(subset=higher, color='#D9EAD3', axis=0)
    if lower:
        styler = styler.highlight_min(subset=lower, color='#D9EAD3', axis=0)
    return styler.hide(axis='index')


def write_sheet(path, sheet, df, index=False):
    path = Path(path)
    kwargs = {'engine': 'openpyxl'}
    if path.exists():
        kwargs.update(mode='a', if_sheet_exists='replace')
    with pd.ExcelWriter(path, **kwargs) as writer:
        df.to_excel(writer, sheet_name=sheet, index=index)




def full_per_class_table(board):
    rows = []
    for r in board.itertuples():
        for cls in CLASSES:
            rows.append({
                'Model': r.Model,
                'Family': r.family,
                'class': cls,
                'AUROC': getattr(r, f'auroc_{cls}'),
                'AUPRC': getattr(r, f'auprc_{cls}'),
                'F1': getattr(r, f'f1_{cls}'),
                'Precision': getattr(r, f'precision_{cls}'),
                'Sensitivity': getattr(r, f'sensitivity_{cls}'),
                'Specificity': getattr(r, f'specificity_{cls}'),
            })
    return pd.DataFrame(rows)


def cm_counts_table(board):
    rows = []
    for r in board.itertuples():
        for cls in CLASSES:
            rows.append({
                'Model': r.Model,
                'class': cls,
                'TN': getattr(r, f'cm_{cls}_tn'),
                'FP': getattr(r, f'cm_{cls}_fp'),
                'FN': getattr(r, f'cm_{cls}_fn'),
                'TP': getattr(r, f'cm_{cls}_tp'),
            })
    return pd.DataFrame(rows)

print('helpers ready')


In [ ]:
board, per_seed = collect_leaderboard(SPLIT)
display(Markdown(f'### Inventory — `{SPLIT}`'))
if board.empty:
    print('No test predictions yet. Re-run Performance analysis in notebooks 01-04 (EVALUATE_TEST=True writes test_predictions.csv), then re-run 05/06.')
else:
    inv = board[['Model', 'family', 'n_seeds', 'macro_auroc', 'macro_auprc', 'macro_f1']].copy()
    display(inv)
    board.to_csv(TABLE_DIR / f'leaderboard_{SPLIT}_raw.csv', index=False)
    per_seed.to_csv(TABLE_DIR / f'per_seed_{SPLIT}.csv', index=False)
    dump_json(
        {'split': SPLIT, 'evaluate_test': EVALUATE_TEST, 'models': board['key'].tolist(), 'n': int(len(board))},
        TABLE_DIR / 'inventory.json',
    )


## Table 1 — Macro leaderboard (members + ensemble)

Best cell **bold + green**, second-best *italic + yellow*. Ensemble rows are tinted. Values are mean ± SD when more than one seed exists.


In [ ]:
if board.empty:
    print('Skip: empty leaderboard')
else:
    show = formatted_leaderboard(board)
    display(Markdown(f'**Table 1.** PTB-XL 5-superclass multi-label, `{SPLIT}` fold. Thresholds = per-class F1-max on validation fold 9.'))
    display(show.style.set_caption('Table 1. Macro metrics').hide(axis='index').set_table_styles([
        {'selector': 'caption', 'props': 'caption-side: top; font-weight: 700; font-size: 13px; text-align: left;'},
        {'selector': 'th', 'props': 'background-color: #1B2838; color: white; padding: 6px 8px;'},
        {'selector': 'td', 'props': 'padding: 5px 8px; text-align: center;'},
    ]))

    numeric_for_hl = board[['macro_auroc', 'macro_auprc', 'macro_f1', 'micro_f1', 'precision', 'recall', 'specificity', 'hamming_loss', 'subset_accuracy']].copy()
    numeric_for_hl.columns = ['Macro AUROC', 'Macro AUPRC', 'Macro F1', 'Micro F1', 'Precision', 'Recall', 'Specificity', 'Hamming ↓', 'Subset acc.']
    draw_journal_table(
        show.drop(columns=['Seeds']),
        title=f'Table 1. All protocol macro metrics ({SPLIT})  ·  mean ± SD',
        stem=f'table01_leaderboard_{SPLIT}',
        numeric_source=numeric_for_hl,
        higher_cols={'Macro AUROC', 'Macro AUPRC', 'Macro F1', 'Micro F1', 'Precision', 'Recall', 'Specificity', 'Subset acc.'},
        lower_cols={'Hamming ↓'},
    )

    # Compact numeric table for Excel / paper markdown
    compact = pd.DataFrame({
        'Model': board['Model'],
        'Family': board['family'],
        'macro_AUROC': board['macro_auroc'].round(4),
        'macro_AUPRC': board['macro_auprc'].round(4),
        'macro_F1': board['macro_f1'].round(4),
        'micro_F1': board['micro_f1'].round(4),
        'Precision': board['precision'].round(4),
        'Recall': board['recall'].round(4),
        'Specificity': board['specificity'].round(4),
        'Hamming_loss': board['hamming_loss'].round(4),
        'subset_accuracy': board['subset_accuracy'].round(4),
        'n_seeds': board['n_seeds'],
    })
    compact.to_csv(TABLE_DIR / f'table01_leaderboard_{SPLIT}.csv', index=False)
    write_sheet(TABLE_DIR / f'comparison_tables_{SPLIT}.xlsx', 'leaderboard', compact)
    display(Markdown('Green = best in column, yellow = second. Hamming is **lower-better**. Ensemble rows have a blue tint. `precision`/`recall`/`specificity` are **macro** averages (aliases of `macro_precision` / `macro_recall` / `macro_specificity`).'))


## Figure 1 — Ranking bars (ensemble vs members)

Horizontal ranking makes the gap between members and ensemble readable at a glance. Hatched bars = ensemble.


In [ ]:
def ranking_panel(ax, board, col, title, xlim=None):
    df = board.sort_values(col, ascending=True).reset_index(drop=True)
    y = np.arange(len(df))
    colors = [MODEL_COLOR[k] for k in df['key']]
    hatches = ['///' if k in ENSEMBLE_KEYS else None for k in df['key']]
    err = df[f'{col}_sd'].to_numpy() if df['n_seeds'].max() > 1 else None
    bars = ax.barh(
        y, df[col], xerr=err, color=colors, edgecolor='#1B2838', linewidth=0.4,
        capsize=3, error_kw={'elinewidth': 0.8, 'ecolor': '#334155'},
    )
    for bar, hatch in zip(bars, hatches):
        bar.set_hatch(hatch)
    ax.set_yticks(y, df['Model'])
    ax.set_xlabel(title)
    if xlim:
        ax.set_xlim(*xlim)
    ax.axvline(df[col].max(), color='#94A3B8', ls=':', lw=0.8, zorder=0)
    for yi, val in zip(y, df[col]):
        ax.text(val + 0.002, yi, f'{val:.3f}', va='center', fontsize=7.5, color='#1B2838')


if not board.empty:
    fig, axes = plt.subplots(1, 3, figsize=(COL_DOUBLE, 0.42 * len(board) + 1.3), sharey=False)
    ranking_panel(axes[0], board, 'macro_auroc', 'Macro AUROC', xlim=(0.80, 1.0) if board['macro_auroc'].min() > 0.8 else None)
    ranking_panel(axes[1], board, 'macro_auprc', 'Macro AUPRC', xlim=(0.55, 0.95) if board['macro_auprc'].min() > 0.55 else None)
    ranking_panel(axes[2], board, 'macro_f1', 'Macro F1', xlim=(0.55, 0.90) if board['macro_f1'].min() > 0.55 else None)
    fig.suptitle(f'Figure 1. Ranking on {SPLIT}  ·  hatched = ensemble', fontsize=11, fontweight='bold', x=0.01, ha='left')
    fig.tight_layout()
    save_fig(fig, f'fig01_ranking_{SPLIT}')
    display(fig)
    plt.close(fig)

    # Grouped metric bars
    metrics = [('macro_auroc', 'AUROC'), ('macro_auprc', 'AUPRC'), ('macro_f1', 'F1'), ('micro_f1', 'micro-F1')]
    fig, ax = plt.subplots(figsize=(COL_DOUBLE, 4.2))
    x = np.arange(len(board))
    width = 0.2
    palette = ['#0072B2', '#D55E00', '#009E73', '#CC79A7']
    for i, (col, lab) in enumerate(metrics):
        ax.bar(
            x + (i - 1.5) * width, board[col], width, label=lab, color=palette[i],
            edgecolor='white', linewidth=0.3,
            yerr=board[f'{col}_sd'] if board['n_seeds'].max() > 1 else None, capsize=2,
        )
    ax.set_xticks(x, board['Model'], rotation=22, ha='right')
    ax.set_ylabel('Score')
    ax.set_ylim(0.5, 1.0)
    ax.legend(ncol=4, frameon=False, loc='upper left')
    ax.set_title(f'Figure 2. Macro metrics side-by-side ({SPLIT})', loc='left', fontweight='bold')
    fig.tight_layout()
    save_fig(fig, f'fig02_grouped_metrics_{SPLIT}')
    display(fig)
    plt.close(fig)

    radar_cols = ['macro_auroc', 'macro_auprc', 'macro_f1', 'precision', 'recall', 'specificity']
    radar_labs = ['AUROC', 'AUPRC', 'F1', 'Prec.', 'Recall', 'Spec.']
    angles = np.linspace(0, 2 * np.pi, len(radar_cols), endpoint=False)
    angles = np.concatenate([angles, angles[:1]])
    fig, ax = plt.subplots(figsize=(COL_SINGLE * 1.7, COL_SINGLE * 1.7), subplot_kw={'polar': True})
    for _, r in board.iterrows():
        vals = [r[c] for c in radar_cols]
        vals.append(vals[0])
        ls = '--' if r['key'] in ENSEMBLE_KEYS else '-'
        ax.plot(angles, vals, color=MODEL_COLOR[r['key']], lw=1.6, ls=ls, label=r['Model'])
        ax.fill(angles, vals, color=MODEL_COLOR[r['key']], alpha=0.05)
    ax.set_xticks(angles[:-1], radar_labs)
    ax.set_ylim(0.5, 1.0)
    ax.legend(bbox_to_anchor=(1.35, 1.05), frameon=False, fontsize=7)
    ax.set_title(f'Figure 2b. Metric profile ({SPLIT})', loc='left', fontweight='bold', pad=16)
    fig.tight_layout()
    save_fig(fig, f'fig02b_radar_{SPLIT}')
    display(fig)
    plt.close(fig)


## Tables 2–4 — Per-class metrics

Every protocol per-class score: AUROC, AUPRC, F1, Precision, Sensitivity (recall), Specificity. Then one **long table** with all six columns together.


In [ ]:
def class_matrix(board, metric):
    cols = [f'{metric}_{c}' for c in CLASSES]
    mat = board.set_index('Model')[cols].copy()
    mat.columns = list(CLASSES)
    return mat


def class_heatmap_and_table(board, metric, pretty, stem, cmap):
    mat = class_matrix(board, metric)
    fig, ax = plt.subplots(figsize=(COL_DOUBLE * 0.62, 0.38 * len(mat) + 1.4))
    vmin = {'auroc': 0.80, 'auprc': 0.55, 'f1': 0.50, 'precision': 0.50, 'sensitivity': 0.50, 'specificity': 0.75}.get(metric)
    sns.heatmap(
        mat, annot=True, fmt='.3f', cmap=cmap, vmin=vmin, vmax=1.0,
        linewidths=0.6, linecolor='white', cbar_kws={'shrink': 0.8, 'label': pretty},
        ax=ax, annot_kws={'fontsize': 8},
    )
    ax.set_xlabel('')
    ax.set_ylabel('')
    ax.set_title(f'{pretty} by class ({SPLIT})', loc='left', fontweight='bold')
    fig.tight_layout()
    save_fig(fig, stem)
    display(fig)
    plt.close(fig)

    show = mat.reset_index()
    display(style_numeric_table(show, f'Table. {pretty} per class', higher=list(CLASSES)))
    draw_journal_table(
        show.round(3),
        title=f'{pretty} per superclass ({SPLIT})',
        stem=f'{stem}_table',
        numeric_source=show,
        higher_cols=set(CLASSES),
    )
    return mat


if not board.empty:
    specs = [
        ('auroc', 'AUROC', 'Blues', f'table02_perclass_auroc_{SPLIT}'),
        ('auprc', 'AUPRC', 'Oranges', f'table03_perclass_auprc_{SPLIT}'),
        ('f1', 'F1', 'Greens', f'table04_perclass_f1_{SPLIT}'),
        ('precision', 'Precision', 'Purples', f'table04b_perclass_precision_{SPLIT}'),
        ('sensitivity', 'Sensitivity', 'YlGnBu', f'table04c_perclass_sensitivity_{SPLIT}'),
        ('specificity', 'Specificity', 'Greys', f'table04d_perclass_specificity_{SPLIT}'),
    ]
    mats = {}
    for key, pretty, cmap, stem in specs:
        mats[key] = class_heatmap_and_table(board, key, pretty, stem, cmap)

    fig, ax = plt.subplots(figsize=(COL_DOUBLE, 4.0))
    melt = mats['auroc'].reset_index().melt(id_vars='Model', var_name='Class', value_name='AUROC')
    sns.barplot(data=melt, x='Class', y='AUROC', hue='Model', palette=[MODEL_COLOR[k] for k in board['key']], ax=ax, edgecolor='white')
    ax.set_ylim(0.75, 1.0)
    ax.legend(bbox_to_anchor=(1.01, 1), loc='upper left', frameon=False, fontsize=7)
    ax.set_title(f'Figure 3. Per-class AUROC ({SPLIT})', loc='left', fontweight='bold')
    fig.tight_layout()
    save_fig(fig, f'fig03_perclass_auroc_bars_{SPLIT}')
    display(fig)
    plt.close(fig)

    long_pc = full_per_class_table(board)
    display(Markdown('**Table 4e.** All per-class protocol metrics in one table (Sensitivity = recall).'))
    display(style_numeric_table(
        long_pc,
        'All per-class metrics',
        higher=['AUROC', 'AUPRC', 'F1', 'Precision', 'Sensitivity', 'Specificity'],
    ))
    draw_journal_table(
        long_pc.round(3),
        title=f'Table 4e. Full per-class metrics ({SPLIT})',
        stem=f'table04e_perclass_all_{SPLIT}',
        numeric_source=long_pc,
        higher_cols={'AUROC', 'AUPRC', 'F1', 'Precision', 'Sensitivity', 'Specificity'},
    )

    xlsx = TABLE_DIR / f'comparison_tables_{SPLIT}.xlsx'
    for key, pretty, _, _ in specs:
        write_sheet(xlsx, f'perclass_{pretty}', mats[key].reset_index())
    write_sheet(xlsx, 'perclass_all', long_pc)


## Table 5 — Ensemble vs members (Δ)

Positive Δ means the ensemble beats that member. Stacking is fit on val fold 9; numbers below are **test fold 10** with frozen val-tuned stacker and thresholds.


In [ ]:
if board.empty:
    print('Skip delta table')
else:
    members = board[board['family'] == 'Member'].copy()
    ensembles = board[board['family'] == 'Ensemble'].copy()
    best_member_key = members.loc[members['macro_auprc'].idxmax(), 'key']
    best_member_name = display_model_name(best_member_key)
    resnet = members.loc[members['key'] == 'resnet18']

    delta_rows = []
    for _, ens in ensembles.iterrows():
        row = {'Ensemble': ens['Model']}
        for col, lab in [('macro_auroc', 'AUROC'), ('macro_auprc', 'AUPRC'), ('macro_f1', 'F1')]:
            row[f'{lab}'] = ens[col]
            if not resnet.empty:
                row[f'Δ {lab} vs ResNet18'] = ens[col] - float(resnet[col].iloc[0])
            row[f'Δ {lab} vs best member'] = ens[col] - float(members[col].max())
        delta_rows.append(row)
    delta = pd.DataFrame(delta_rows)
    display(Markdown(f'Best member on this split (**macro-AUPRC**): **{best_member_name}**'))
    display(style_numeric_table(
        delta,
        'Table 5. Ensemble minus ResNet18-1D and minus best individual member',
        higher=[c for c in delta.columns if c != 'Ensemble'],
    ))
    draw_journal_table(
        delta.round(3),
        title=f'Table 5. Ensemble gain ({SPLIT})  ·  best member = {best_member_name}',
        stem=f'table05_ensemble_delta_{SPLIT}',
        numeric_source=delta.drop(columns=['Ensemble']),
        higher_cols=set(delta.columns) - {'Ensemble'},
    )

    # Member-level gain heatmap for the strongest ensemble method
    if not ensembles.empty:
        top_ens = ensembles.loc[ensembles['macro_auprc'].idxmax()]
        gain = pd.DataFrame(index=members['Model'], columns=['Δ AUROC', 'Δ AUPRC', 'Δ F1'], dtype=float)
        for _, m in members.iterrows():
            gain.loc[m['Model'], 'Δ AUROC'] = top_ens['macro_auroc'] - m['macro_auroc']
            gain.loc[m['Model'], 'Δ AUPRC'] = top_ens['macro_auprc'] - m['macro_auprc']
            gain.loc[m['Model'], 'Δ F1'] = top_ens['macro_f1'] - m['macro_f1']
        fig, ax = plt.subplots(figsize=(COL_SINGLE * 1.35, 0.42 * len(gain) + 1.2))
        sns.heatmap(gain, annot=True, fmt='+.3f', cmap='RdBu_r', center=0.0, ax=ax, linewidths=0.6, linecolor='white')
        ax.set_title(f'Figure 4. {top_ens["Model"]} minus each member', loc='left', fontweight='bold')
        fig.tight_layout()
        save_fig(fig, f'fig04_ensemble_gain_{SPLIT}')
        display(fig)
        plt.close(fig)
        xlsx = TABLE_DIR / f'comparison_tables_{SPLIT}.xlsx'
        write_sheet(xlsx, 'ensemble_delta', delta)
        write_sheet(xlsx, 'ensemble_vs_members', gain.reset_index())


## Table 6 — Literature (Strodthoff et al. 2021) vs this work

Published numbers are **test-fold 10, 100 Hz, official `strat_fold`, 5-superclass (superdiagnostic)** from Strodthoff et al., *IEEE JBHI* 2021, Table II (macro AUROC and F-max). They are **not** retrained here.

This work uses the same split protocol and reports **official test fold 10**. Thresholds and ensemble weights stay val-tuned (fold 9). F-max (literature) and macro-F1 (this work, val-tuned thresholds) are related but not identical — do not treat them as one metric.


In [ ]:
# Strodthoff et al., IEEE JBHI 2021 Table II — PTB-XL superdiagnostic (5 classes).
# Values: macro AUROC (95% bootstrap interval half-width in the paper, shown as ±).
LIT = pd.DataFrame([
    {'Method': 'Naive', 'Source': 'Strodthoff 2021', 'kind': 'published', 'macro_AUROC': 0.500, 'AUROC_pm': 0.000, 'Fmax': 0.448},
    {'Method': 'Wavelet+NN', 'Source': 'Strodthoff 2021', 'kind': 'published', 'macro_AUROC': 0.874, 'AUROC_pm': 0.008, 'Fmax': 0.734},
    {'Method': 'Inception1d', 'Source': 'Strodthoff 2021', 'kind': 'published', 'macro_AUROC': 0.921, 'AUROC_pm': 0.006, 'Fmax': 0.810},
    {'Method': 'LSTM-bidir', 'Source': 'Strodthoff 2021', 'kind': 'published', 'macro_AUROC': 0.921, 'AUROC_pm': 0.006, 'Fmax': 0.815},
    {'Method': 'FCN-Wang', 'Source': 'Strodthoff 2021', 'kind': 'published', 'macro_AUROC': 0.925, 'AUROC_pm': 0.006, 'Fmax': 0.817},
    {'Method': 'LSTM', 'Source': 'Strodthoff 2021', 'kind': 'published', 'macro_AUROC': 0.927, 'AUROC_pm': 0.005, 'Fmax': 0.820},
    {'Method': 'xResNet1d101', 'Source': 'Strodthoff 2021', 'kind': 'published', 'macro_AUROC': 0.928, 'AUROC_pm': 0.006, 'Fmax': 0.815},
    {'Method': 'ResNet1d-Wang', 'Source': 'Strodthoff 2021', 'kind': 'published', 'macro_AUROC': 0.930, 'AUROC_pm': 0.006, 'Fmax': 0.823},
    {'Method': 'Ensemble (Strodthoff)', 'Source': 'Strodthoff 2021', 'kind': 'published', 'macro_AUROC': 0.934, 'AUROC_pm': 0.005, 'Fmax': 0.825},
])

ours = []
if not board.empty:
    for _, r in board.iterrows():
        ours.append({
            'Method': r['Model'],
            'Source': f'This work ({SPLIT})',
            'kind': 'this_work',
            'macro_AUROC': r['macro_auroc'],
            'AUROC_pm': r['macro_auroc_sd'] if r['n_seeds'] > 1 else 0.0,
            'Fmax': r['macro_f1'],
        })
ours_df = pd.DataFrame(ours)
combined = pd.concat([LIT, ours_df], ignore_index=True)

lit_show = LIT.copy()
lit_show['macro AUROC'] = lit_show.apply(lambda r: f"{r.macro_AUROC:.3f} ± {r.AUROC_pm:.3f}", axis=1)
lit_show['F-max'] = lit_show['Fmax'].map(lambda x: f'{x:.3f}')
lit_view = lit_show[['Method', 'Source', 'macro AUROC', 'F-max']]
display(Markdown('**Table 6a.** Published PTB-XL superdiagnostic scores (test fold 10).'))
display(lit_view.style.hide(axis='index').set_table_styles([
    {'selector': 'th', 'props': 'background-color: #1B2838; color: white; padding: 6px 8px;'},
    {'selector': 'td', 'props': 'padding: 5px 8px; text-align: center;'},
]))
draw_journal_table(
    lit_view,
    title='Table 6a. Strodthoff et al. 2021  ·  superdiagnostic test AUROC / F-max',
    stem='table06a_literature_strodthoff',
    numeric_source=LIT.rename(columns={'macro_AUROC': 'macro AUROC', 'Fmax': 'F-max'})[['macro AUROC', 'F-max']],
    higher_cols={'macro AUROC', 'F-max'},
)

if not ours_df.empty:
    ours_view = pd.DataFrame({
        'Method': ours_df['Method'],
        'Source': ours_df['Source'],
        'macro AUROC': ours_df.apply(lambda r: fmt_mean_sd(r.macro_AUROC, r.AUROC_pm, 2 if r.AUROC_pm else 1), axis=1),
        'macro F1': ours_df['Fmax'].map(lambda x: f'{x:.3f}'),
    })
    display(Markdown(f'**Table 6b.** This work on `{SPLIT}` (same 5 superclasses, official folds).'))
    display(ours_view.style.hide(axis='index').set_table_styles([
        {'selector': 'th', 'props': 'background-color: #1B2838; color: white; padding: 6px 8px;'},
        {'selector': 'td', 'props': 'padding: 5px 8px; text-align: center;'},
    ]))
    draw_journal_table(
        ours_view,
        title=f'Table 6b. This work ({SPLIT})  ·  5-superclass multi-label',
        stem=f'table06b_this_work_{SPLIT}',
        numeric_source=ours_df.rename(columns={'macro_AUROC': 'macro AUROC', 'Fmax': 'macro F1'})[['macro AUROC', 'macro F1']],
        higher_cols={'macro AUROC', 'macro F1'},
    )

# Overlay lollipop
fig, ax = plt.subplots(figsize=(COL_DOUBLE, 0.32 * len(combined) + 1.6))
plot_df = combined.sort_values('macro_AUROC').reset_index(drop=True)
y = np.arange(len(plot_df))
for i, r in plot_df.iterrows():
    color = '#64748B' if r['kind'] == 'published' else MODEL_COLOR.get(
        next((k for k in ALL_KEYS if display_model_name(k) == r['Method']), 'resnet18'),
        '#0072B2',
    )
    marker = 'o' if r['kind'] == 'published' else 's'
    ax.hlines(i, 0.48, r['macro_AUROC'], color=color, lw=1.2, alpha=0.85)
    ax.errorbar(
        r['macro_AUROC'], i, xerr=r['AUROC_pm'] if r['AUROC_pm'] else None,
        fmt=marker, color=color, ms=7, mec='#1B2838', mew=0.4, capsize=2, ecolor=color,
    )
ax.set_yticks(y, [f"{r.Method}  ·  {r.Source}" for r in plot_df.itertuples()])
ax.set_xlabel('Macro AUROC')
ax.set_xlim(0.48, 0.98)
ax.set_title(
    f'Figure 5. Literature (circles) vs this work (squares, {SPLIT})',
    loc='left', fontweight='bold',
)
ax.axvline(0.930, color='#94A3B8', ls='--', lw=0.7, label='Strodthoff ResNet1d-Wang 0.930')
ax.legend(frameon=False, loc='lower right')
fig.tight_layout()
save_fig(fig, f'fig05_literature_overlay_{SPLIT}')
display(fig)
plt.close(fig)

combined.to_csv(TABLE_DIR / f'table06_literature_vs_ours_{SPLIT}.csv', index=False)
write_sheet(TABLE_DIR / f'comparison_tables_{SPLIT}.xlsx', 'literature', combined)

display(Markdown(
    'Citation: Strodthoff N, Wagner P, Schaeffter T, Samek W. Deep Learning for ECG Analysis: '
    'Benchmarks and Insights from PTB-XL. IEEE J Biomed Health Inform. 2021;25(5):1519–1528. '
    'doi:10.1109/JBHI.2020.3022989. Superdiagnostic columns of Table II.'
))


## Figure 6–7 — ROC and precision–recall (micro-pooled)

Includes ensemble methods. Micro-pooled over the five superclasses.


In [ ]:
rep_seed = 42 if 42 in SEEDS else SEEDS[0]
bundle = ensemble_bundle(rep_seed)

fig_roc, ax_roc = plt.subplots(figsize=(COL_SINGLE * 1.55, COL_SINGLE * 1.55))
fig_pr, ax_pr = plt.subplots(figsize=(COL_SINGLE * 1.55, COL_SINGLE * 1.55))

for key in ALL_KEYS:
    loaded = load_model_arrays(key, rep_seed, SPLIT, bundle=bundle)
    if loaded is None:
        continue
    y_true, y_prob, _ = loaded
    y_t, y_s = y_true.ravel(), y_prob.ravel()
    fpr, tpr, _ = roc_curve(y_t, y_s)
    prec, rec, _ = precision_recall_curve(y_t, y_s)
    ls = '--' if key in ENSEMBLE_KEYS else '-'
    lw = 2.0 if key in ENSEMBLE_KEYS else 1.4
    ax_roc.plot(fpr, tpr, color=MODEL_COLOR[key], ls=ls, lw=lw,
                label=f'{display_model_name(key)}  {auc(fpr, tpr):.3f}')
    ax_pr.plot(rec, prec, color=MODEL_COLOR[key], ls=ls, lw=lw, label=display_model_name(key))

ax_roc.plot([0, 1], [0, 1], color='#94A3B8', ls=':', lw=0.8)
ax_roc.set_xlabel('False positive rate')
ax_roc.set_ylabel('True positive rate')
ax_roc.set_title(f'Figure 6. ROC (micro-pooled, {SPLIT}, seed {rep_seed})', loc='left', fontweight='bold')
ax_roc.legend(frameon=False, fontsize=7, loc='lower right')
ax_roc.set_xlim(0, 1)
ax_roc.set_ylim(0, 1)

ax_pr.set_xlabel('Recall')
ax_pr.set_ylabel('Precision')
ax_pr.set_title(f'Figure 7. Precision–recall (micro-pooled, {SPLIT}, seed {rep_seed})', loc='left', fontweight='bold')
ax_pr.legend(frameon=False, fontsize=7, loc='lower left')
ax_pr.set_xlim(0, 1)
ax_pr.set_ylim(0, 1)

fig_roc.tight_layout()
fig_pr.tight_layout()
save_fig(fig_roc, f'fig06_roc_micro_{SPLIT}')
save_fig(fig_pr, f'fig07_pr_micro_{SPLIT}')
display(fig_roc)
display(fig_pr)
plt.close(fig_roc)
plt.close(fig_pr)


## Figure 8 — Member diversity (test probabilities)

Pairwise mean-probability correlation, disagreement, and error-overlap Jaccard on **test fold 10**. High correlation / high error overlap ⇒ less complementary members.


In [ ]:
div_seed = 42 if 42 in SEEDS else SEEDS[0]
split_frames = load_member_frames(div_seed, SPLIT)
if split_frames is None:
    print('Need all five member test_predictions.csv for diversity.')
else:
    _, split_probs, y_split = align_prediction_frames(split_frames, CLASSES)
    labels = [display_model_name(k) for k in MEMBER_KEYS]
    corr = pairwise_prediction_correlation(split_probs, MEMBER_KEYS)
    corr.index = labels
    corr.columns = labels
    pred_bin = {}
    for key in MEMBER_KEYS:
        loaded = load_model_arrays(key, div_seed, SPLIT)
        pred_bin[key] = apply_thresholds(loaded[1], loaded[2])
    disc = disagreement_rate(pred_bin, MEMBER_KEYS)
    disc.index = labels
    disc.columns = labels
    ov = error_overlap(y_split, pred_bin, MEMBER_KEYS)
    ov.index = labels
    ov.columns = labels

    fig, axes = plt.subplots(1, 3, figsize=(COL_DOUBLE, 3.3))
    for ax, mat, title, cmap, fmt in [
        (axes[0], corr, 'Probability correlation', 'mako', '.2f'),
        (axes[1], disc, 'Disagreement rate', 'YlOrBr', '.2f'),
        (axes[2], ov, 'Error overlap (Jaccard)', 'rocket', '.2f'),
    ]:
        sns.heatmap(mat, annot=True, fmt=fmt, cmap=cmap, ax=ax, square=True, linewidths=0.4, linecolor='white', annot_kws={'fontsize': 7})
        ax.set_title(title, fontweight='bold', fontsize=9)
    fig.suptitle(f'Figure 8. Diversity on {SPLIT} (seed {div_seed})', fontsize=11, fontweight='bold', x=0.01, ha='left')
    fig.tight_layout()
    save_fig(fig, f'fig08_diversity_{SPLIT}')
    display(fig)
    plt.close(fig)

    display(Markdown(f'**Table 7.** Pairwise probability correlation ({SPLIT})'))
    display(style_numeric_table(corr.reset_index().rename(columns={'index': 'Model'}), 'Correlation', lower=labels))
    corr.to_csv(TABLE_DIR / f'table07_diversity_correlation_{SPLIT}.csv')
    disc.to_csv(TABLE_DIR / f'table07_diversity_disagreement_{SPLIT}.csv')
    ov.to_csv(TABLE_DIR / f'table07_diversity_error_overlap_{SPLIT}.csv')


## Figure 9 — Per-class 2×2 confusion matrices

One row per model (ensemble included). Columns = five superclasses. Not a 5×5 multiclass matrix.


In [ ]:
cm_seed = 42 if 42 in SEEDS else SEEDS[0]
bundle = ensemble_bundle(cm_seed)
cm_items = []
for key in ALL_KEYS:
    loaded = load_model_arrays(key, cm_seed, SPLIT, bundle=bundle)
    if loaded is None:
        continue
    y_true, y_prob, thr = loaded
    y_pred = apply_thresholds(y_prob, thr)
    cm_items.append((key, y_true, y_pred))

if not cm_items:
    print('No predictions for confusion matrices.')
else:
    n_m = len(cm_items)
    fig, axes = plt.subplots(n_m, 5, figsize=(COL_DOUBLE, 1.28 * n_m + 0.4))
    if n_m == 1:
        axes = np.array([axes])
    for r, (key, y_true, y_pred) in enumerate(cm_items):
        cms = per_class_confusion_2x2(y_true, y_pred, CLASSES)
        for c, name in enumerate(CLASSES):
            ax = axes[r, c]
            mat = np.array([[cms[name]['tn'], cms[name]['fp']], [cms[name]['fn'], cms[name]['tp']]], dtype=int)
            ax.imshow(mat, cmap='Blues', vmin=0, vmax=mat.max() if mat.max() else 1)
            ax.set_xticks([0, 1], ['P0', 'P1'], fontsize=6)
            ax.set_yticks([0, 1], ['T0', 'T1'], fontsize=6)
            for (i, j), val in np.ndenumerate(mat):
                ax.text(j, i, str(val), ha='center', va='center', fontsize=7, color='#0F172A')
            if r == 0:
                ax.set_title(name, fontsize=8, fontweight='bold')
            if c == 0:
                ax.set_ylabel(display_model_name(key), fontsize=7, fontweight='semibold')
    fig.suptitle(f'Figure 9. Per-class 2×2 CM ({SPLIT}, seed {cm_seed})', fontsize=11, fontweight='bold', x=0.01, ha='left')
    fig.tight_layout()
    save_fig(fig, f'fig09_cm2x2_grid_{SPLIT}')
    display(fig)
    plt.close(fig)

if not board.empty:
    cm_tab = cm_counts_table(board)
    display(Markdown(f'**Table 8.** Per-class 2×2 counts ({SPLIT}; mean across seeds if n>1).'))
    display(cm_tab.style.format(precision=1).hide(axis='index').set_table_styles([
        {'selector': 'th', 'props': 'background-color: #1B2838; color: white; padding: 6px 8px;'},
        {'selector': 'td', 'props': 'padding: 5px 8px; text-align: center;'},
    ]))
    draw_journal_table(
        cm_tab.round(0),
        title=f'Table 8. Confusion counts TN/FP/FN/TP ({SPLIT})',
        stem=f'table08_cm_counts_{SPLIT}',
    )
    write_sheet(TABLE_DIR / f'comparison_tables_{SPLIT}.xlsx', 'cm_counts', cm_tab)
    cm_tab.to_csv(TABLE_DIR / f'table08_cm_counts_{SPLIT}.csv', index=False)


## Exports

PNG + PDF → `paper/results_figures/`. CSV + Excel → `paper/results_tables/`. A copy of PNGs also goes to `experiments/comparison/latest/`.


In [ ]:
print('figures', FIG_DIR)
for p in sorted(FIG_DIR.glob('*')):
    print(' ', p.name)
print('tables', TABLE_DIR)
for p in sorted(TABLE_DIR.glob('*')):
    print(' ', p.name)
print('done')
